# Subdivision lab

Builds an L-mesh on `data/triMeshTest.obj` (sphere centres along the cone axes, radius = ½ the
smallest curvature-sphere radius), subdivides it (`subdivide_lmesh`) and shows it. The last cell
exports the result to `data/lmeshes/subdivided_lmesh`.

**Viewer:** `ws.init()` starts a kayviz viewer and prints its URL; open it in a browser. To share one viewer between several notebooks, run `python -m kayviz` in a terminal first.

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
for sub in ("",):  # project root: LMesh/, Lconjugacy/ (hanan, kayviz are installed)
    p = os.path.join(PROJECT_ROOT, sub) if sub else PROJECT_ROOT
    if p not in sys.path:
        sys.path.insert(0, p)

print("Project root:", PROJECT_ROOT)

In [ ]:
import numpy as np

from kayviz.webscope import script as ws
from kayviz import serializers as ser

from LMesh.L_mesh import Lmesh
from hanan.geometry.utils import (
    read_obj,
    normalize_vertices,
    write_obj,
    compute_cone_axes,
    conical_mesh_smallest_curvature_sphere,
)
from hanan.geometry.mesh import Mesh

In [ ]:
data_folder = os.path.join(PROJECT_ROOT, 'data')
mesh_name   = 'triMeshTest.obj'

V, F = read_obj(os.path.join(data_folder, mesh_name))
V    = normalize_vertices(V)

base_mesh = Mesh()
base_mesh.make_mesh(V, F)

inn_vertices = base_mesh.inner_vertices()

In [ ]:
axes, angles       = compute_cone_axes(base_mesh)
v_adjacency_list   = base_mesh.vertex_adjacency_list()
centers_int, smallest_radii = conical_mesh_smallest_curvature_sphere(
    base_mesh.vertices, axes, angles, v_adjacency_list
)

radius  = 0.5 * np.abs(smallest_radii)
centers = V + radius[:, None] * axes

In [ ]:
lmesh = Lmesh()
lmesh.init_L_mesh(base_mesh, orientation=1, combinatorics='primal',
                  mode='SphCongruence',
                  radii=radius,
                  centers=centers)
lmesh.improve_contact()

lmesh.subdivide_lmesh()
lmesh.improve_contact()


In [ ]:
ws.init()

ws.register_surface_mesh('Base Mesh', V, F, color=(0.5, 0.5, 0.5))
ws.add_vector_quantity('Base Mesh', 'Cone Axes', axes, defined_on='vertices')
from LMesh.lmesh_serializers import lmesh_scene
ws.push_objects(lmesh_scene(lmesh, prefix='Subdivided L-mesh'))

ws.show()

In [ ]:
lmesh.export_data(os.path.join(data_folder, 'lmeshes', 'subdivided_lmesh'))